# Recurrence from first principles

**Learning objective:** Compute the hidden-state recurrence manually and connect the equation to a TensorFlow `SimpleRNNCell`.

This notebook is part of the TensorFlow/Keras learning track. It is designed to be read top-to-bottom: intuition → shapes → mathematics → TensorFlow implementation → observed result → interpretation.

> GitHub renders the committed executed output as a static learning artifact. Clone the repository and rerun it in Jupyter/VS Code for live experimentation.


In [1]:
import os, warnings, random
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "bundled with TensorFlow")
print("Execution device(s):", [d.device_type for d in tf.config.list_logical_devices()])


2026-09-21 07:13:15.578004: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789974795.593191    2799 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789974795.597466    2799 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.18.1
Keras: 3.15.1
Execution device(s): ['CPU']


2026-09-21 07:13:17.237673: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


For a vanilla RNN, one common update is
        \[h_t = \tanh(x_tW_x + h_{t-1}W_h + b).\]
        The same weights are reused at every timestep. This **weight sharing through time** is the defining recurrence.


In [2]:
xs = np.array([[1.0],[0.5],[-0.25]], dtype=np.float32)
Wx = np.array([[0.8]], dtype=np.float32); Wh = np.array([[0.4]], dtype=np.float32); b = np.array([0.1], dtype=np.float32)
h = np.array([0.0], dtype=np.float32); states=[]
for step, xt in enumerate(xs):
    h = np.tanh(xt @ Wx + h @ Wh + b)
    states.append(float(h[0])); print(f"t={step}: x={xt[0]: .2f} -> h={h[0]: .4f}")


t=0: x= 1.00 -> h= 0.7163
t=1: x= 0.50 -> h= 0.6564
t=2: x=-0.25 -> h= 0.1612


In [3]:
cell = tf.keras.layers.SimpleRNNCell(1, activation="tanh")
_ = cell(tf.constant([[0.0]]), [tf.constant([[0.0]])])
cell.set_weights([Wx, Wh, b])
h_tf = tf.zeros((1,1))
tf_states=[]
for xt in xs:
    out, [h_tf] = cell(tf.constant(xt.reshape(1,1)), [h_tf]); tf_states.append(float(h_tf.numpy()[0,0]))
print("NumPy states:", np.round(states,6)); print("TensorFlow states:", np.round(tf_states,6)); print("match:", np.allclose(states, tf_states))


NumPy states: [0.716298 0.656433 0.161156]
TensorFlow states: [0.716298 0.656433 0.161156]
match: True


The match demonstrates that Keras is applying the same recurrence we computed explicitly; the framework automates tensor operations and differentiation, not the underlying idea.
